# 07. Operating Proof And Trends

This notebook is the M6 design-partner operations lane.

It shows the live pilot as an operating surface rather than only a semantic demo:

- authenticated service status with namespace and storage context
- canonical coordination report counts
- cut-diff report evidence between `AsOf` and `Current`
- tuple-trace drill-down from a report row
- audit intent fields that show which operator surfaces were used
- local benchmark trend artifacts when they exist in the checkout

In [ ]:
from pathlib import Path
import subprocess
import sys

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(
    (
        path
        for path in candidate_roots
        if (path / "python").exists() and (path / "Cargo.toml").exists()
    ),
    None,
)

if repo_root is None:
    repo_root = Path("/content/AETHER")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/fyremael/AETHER", str(repo_root)],
            check=True,
        )

python_root = repo_root / "python"
if str(python_root) not in sys.path:
    sys.path.insert(0, str(python_root))

repo_root

In [ ]:
from notebooks.colab_setup import ensure_rust_toolchain, pretty_json, start_pilot_service, stop_http_service
from aether_sdk import (
    AetherClient,
    COORDINATION_PILOT_PRE_HEARTBEAT_ELEMENT,
    coordination_pilot_seed_history,
    make_policy_context,
)

ensure_rust_toolchain()
session = start_pilot_service(repo_root)
client = AetherClient(
    session.base_url,
    bearer_token=session.bearer_token,
    namespace=session.namespace,
)

print(f"Pilot namespace: {session.namespace}")
pretty_json(client.status())

## Seed The Canonical Coordination Pilot

The seed history uses the literal v1 operation vocabulary: `Claim`, `LeaseOpen`, `LeaseRenew`, and `Annotate`.
That matters because the report surfaces are proving the actual semantic contract, not a simplified story.

In [ ]:
client.append(coordination_pilot_seed_history())

history = client.history()
status = client.status()
pretty_json(
    {
        "history_len": len(history["datoms"]),
        "effective_namespace": status["effective_namespace"],
        "storage_backend": status["storage"]["backend"],
        "token_id": status["principals"][0]["token_id"],
    }
)

## Live Operator Reports

The report endpoint returns structured JSON, not markdown scraped by a notebook.
The same server-side report builder feeds saved artifacts, HTTP, and the TUI.

In [ ]:
operator_policy = make_policy_context(capabilities=["executor"])
report = client.coordination_report(policy_context=operator_policy)

sections = [
    "pre_heartbeat_authorized",
    "as_of_authorized",
    "live_heartbeats",
    "current_authorized",
    "claimable",
    "accepted_outcomes",
    "rejected_outcomes",
]
pretty_json({section: len(report[section]) for section in sections})

In [ ]:
delta = client.coordination_delta_report(
    left={"kind": "as_of", "element": COORDINATION_PILOT_PRE_HEARTBEAT_ELEMENT},
    right={"kind": "current"},
    policy_context=operator_policy,
)

delta_summary = {
    section: {
        "added": len(delta[section]["added"]),
        "removed": len(delta[section]["removed"]),
        "changed": len(delta[section]["changed"]),
    }
    for section in [
        "current_authorized",
        "claimable",
        "live_heartbeats",
        "accepted_outcomes",
        "rejected_outcomes",
    ]
}
pretty_json(delta_summary)

## Drill Down From Row To Proof

Rows with tuple IDs can be explained directly.
That gives the operator path: report row -> tuple trace -> source datoms and parent tuples.

In [ ]:
row = report["current_authorized"][0]
print(f"Explaining report tuple_id={row['tuple_id']}")
pretty_json(client.explain_tuple(row["tuple_id"], policy_context=operator_policy))

## Audit Intent

M6 adds operator intent fields to audit context: command/source, selected report, selected cut, namespace, token, and principal identity.

In [ ]:
audit = client.audit_log()
interesting = []
for entry in audit["entries"][-8:]:
    context = entry.get("context", {})
    interesting.append(
        {
            "path": entry.get("path"),
            "status": entry.get("status"),
            "principal": entry.get("principal"),
            "namespace": context.get("namespace"),
            "command_source": context.get("command_source"),
            "selected_report": context.get("selected_report"),
            "selected_cut": context.get("selected_cut"),
        }
    )
pretty_json(interesting)

## Benchmark Trend Artifacts

Colab is not the canonical performance lab, but it can show whether this checkout already has trend artifacts.
For real gates, run the host-aware benchmark scripts on the target host or CI runner.

In [ ]:
trend_report = repo_root / "artifacts" / "performance" / "trends" / "latest.md"
if trend_report.exists():
    print("Found local trend report:", trend_report)
    print("\n".join(trend_report.read_text(encoding="utf-8").splitlines()[:30]))
else:
    print("No local trend report found in this checkout.")
    print("Generate one on Windows with:")
    print(r"powershell -NoProfile -ExecutionPolicy Bypass -File scripts\run-performance-trends.ps1")

## What This Proves

The notebook path now reaches the same operator truth surfaces as the packaged pilot:
status, report, cut diff, proof trace, audit context, and benchmark trend evidence.

It still does not claim distributed platform readiness.
It demonstrates the M6-A posture: exact single-node pilot truth with visible operating evidence.

In [ ]:
# Uncomment this cell when you are done with the notebook.
# stop_http_service(session)